In [ ]:
# ==========================================
# External Validation of RF Model
# ==========================================

import pandas as pd
import numpy as np
import joblib

from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.Chem import rdFingerprintGenerator
from rdkit import DataStructs

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)

def smiles_to_inchikey(smiles):

    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return None

    return Chem.MolToInchiKey(mol)
# ==========================================
# Load Training Dataset
# ==========================================

train_df = pd.read_csv("../data/merged_dataset.csv")

print(f"Training dataset size: {len(train_df):,}")

# ==========================================
# Load External Dataset
# ==========================================

external_df = pd.read_excel(
    "../data/MMV_JH.xlsx"
)

print(f"Original external dataset size: {len(external_df):,}")

# ==========================================
# Remove Missing SMILES
# ==========================================

external_df = external_df.dropna(
    subset=["Smiles"]
)

# ==========================================
# Remove Compounds Already Seen During Training
# ==========================================

train_df["inchikey"] = train_df[
    "canonical_smiles"
].apply(smiles_to_inchikey)

external_df["inchikey"] = external_df[
    "Smiles"
].apply(smiles_to_inchikey)

external_df = external_df[
    ~external_df["inchikey"].isin(
        train_df["inchikey"]
    )
]
print(
    f"External dataset after removing training compounds: {len(external_df):,}"
)

# ==========================================
# Remove Duplicate SMILES Within External Set
# ==========================================

external_df = external_df.drop_duplicates(
    subset=["Smiles"]
)

print(
    f"External dataset after removing duplicates: {len(external_df):,}"
)

# ==========================================
# Feature Generation Functions
# ==========================================

def calculate_descriptors(mol):

    return [
        Descriptors.MolWt(mol),
        Descriptors.MolLogP(mol),
        Descriptors.TPSA(mol),
        Descriptors.NumHDonors(mol),
        Descriptors.NumHAcceptors(mol),
        Descriptors.NumRotatableBonds(mol),
        Descriptors.RingCount(mol)
    ]

# Morgan generator
morgan_gen = rdFingerprintGenerator.GetMorganGenerator(
    radius=2,
    fpSize=2048
)

# ==========================================
# Generate Features
# ==========================================

features = []
labels = []
valid_smiles = []

for _, row in external_df.iterrows():

    smiles = row["Smiles"]

    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        continue

    # descriptors
    desc = calculate_descriptors(mol)

    # fingerprint
    fp = morgan_gen.GetFingerprint(mol)

    fp_array = np.zeros((2048,), dtype=int)

    DataStructs.ConvertToNumpyArray(
        fp,
        fp_array
    )

    combined = np.concatenate(
        [desc, fp_array]
    )

    features.append(combined)

    labels.append(
        row["Activity"]
    )

    valid_smiles.append(smiles)

# ==========================================
# Convert to Arrays
# ==========================================

X_external = np.array(features)

y_external = np.array(labels)

print(
    f"Valid compounds used for validation: {len(X_external):,}"
)

print(
    f"Feature matrix shape: {X_external.shape}"
)
# Save generated features and labels

# np.save(
#    "../data/external_validation_features.npy",
#    X_external
#)

#np.save(
#    "../data/external_validation_labels.npy",
#    y_external
#)

#print(
#    "External validation features saved successfully."
#)

# ==========================================
# Load Preprocessing Objects
# ==========================================

imputer = joblib.load(
    "../models/imputer.pkl"
)

scaler = joblib.load(
    "../models/scaler.pkl"
)

# Apply same preprocessing
X_external = imputer.transform(
    X_external
)

X_external = scaler.transform(
    X_external
)

# ==========================================
# Load RF Model
# ==========================================

rf_model = joblib.load(
    "../models/random_forest_model.pkl"
)

# ==========================================
# Predictions
# ==========================================

preds = rf_model.predict(
    X_external
)

probs = rf_model.predict_proba(
    X_external
)[:, 1]

# ==========================================
# Evaluation Metrics
# ==========================================

accuracy = accuracy_score(
    y_external,
    preds
)

precision = precision_score(
    y_external,
    preds
)

recall = recall_score(
    y_external,
    preds
)

f1 = f1_score(
    y_external,
    preds
)

auroc = roc_auc_score(
    y_external,
    probs
)

print("\n==============================")
print("EXTERNAL VALIDATION RESULTS")
print("==============================")

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"AUROC    : {auroc:.4f}")

print("\nClassification Report:\n")

print(
    classification_report(
        y_external,
        preds
    )
)

# ==========================================
# Save Predictions
# ==========================================

results_df = pd.DataFrame({
    "canonical_smiles": valid_smiles,
    "true_label": y_external,
    "predicted_label": preds,
    "prediction_probability": probs
})

results_df.to_csv(
    "../results/external_validation_predictions.csv",
    index=False
)

print(
    "\nPredictions saved to:"
)

print(
    "../results/external_validation_predictions.csv"
)

Training dataset size: 33,208
Original external dataset size: 2,524


[08:23:06] WARNING: not removing hydrogen atom without neighbors


External dataset after removing training compounds: 2,308
External dataset after removing duplicates: 2,233


[08:23:25] WARNING: not removing hydrogen atom without neighbors


Valid compounds used for validation: 2,233
Feature matrix shape: (2233, 2055)

EXTERNAL VALIDATION RESULTS
Accuracy : 0.9019
Precision: 0.0882
Recall   : 0.0157
F1 Score : 0.0267
AUROC    : 0.5922

Classification Report:

              precision    recall  f1-score   support

           0       0.91      0.98      0.95      2042
           1       0.09      0.02      0.03       191

    accuracy                           0.90      2233
   macro avg       0.50      0.50      0.49      2233
weighted avg       0.84      0.90      0.87      2233


Predictions saved to:
../results/external_validation_predictions.csv
